In [119]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [121]:
def process_all_pdf(pdf_directory):
    """processing all PDF"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nprocessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents: #Addition of some more meta data info 
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    return all_documents

all_pdf_documents=process_all_pdf("../data")

Found 7 PDF files to process

processing:Uni_Var_Time_Series_Lecture_1.pdf
Loaded 14 pages

processing:Uni_Var_Time_Series_Lecture_10.pdf
Loaded 16 pages

processing:Uni_Var_Time_Series_Lecture_2..pdf
Loaded 15 pages

processing:Uni_Var_Time_Series_Lecture_4..pdf
Loaded 12 pages

processing:Uni_Var_Time_Series_Lecture_6.pdf
Loaded 15 pages

processing:Uni_Var_Time_Series_Lecture_7A.pdf
Loaded 12 pages

processing:Uni_Var_Time_Series_Lecture_8.pdf
Loaded 12 pages


In [122]:
all_pdf_documents

[Document(metadata={'producer': 'MiKTeX pdfTeX-1.40.24', 'creator': 'LaTeX with Beamer class', 'creationdate': '2023-07-28T08:08:51+05:30', 'author': 'T. V. Ramanathan', 'title': 'Modeling Univariate Time Series', 'subject': '', 'keywords': '', 'moddate': '2023-07-28T08:08:51+05:30', 'trapped': '/False', 'ptex.fullbanner': 'This is MiKTeX-pdfTeX 4.11.0 (1.40.24)', 'source': '..\\data\\Time series\\Uni_Var_Time_Series_Lecture_1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'Uni_Var_Time_Series_Lecture_1.pdf', 'file_type': 'pdf'}, page_content='Modeling Univariate Time Series\nT. V. Ramanathan\nDepartment of Statistics\nSavitribai Phule Pune University\nPune - 411007 (India).\nram@unipune.ac.in\nLecture - 1\nJuly 28, 2023\nT. V. Ramanathan Univariate Time Series Analysis'),
 Document(metadata={'producer': 'MiKTeX pdfTeX-1.40.24', 'creator': 'LaTeX with Beamer class', 'creationdate': '2023-07-28T08:08:51+05:30', 'author': 'T. V. Ramanathan', 'title': 'Modeling Univ

In [123]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")

    return split_docs

In [124]:
chunks=split_documents(all_pdf_documents,chunk_size=200)

Split 96 documents into 497 chunks

Example chunk:
Content: Modeling Univariate Time Series
T. V. Ramanathan
Department of Statistics
Savitribai Phule Pune University
Pune - 411007 (India).
ram@unipune.ac.in
Lecture - 1
July 28, 2023...
Metadata:{'producer': 'MiKTeX pdfTeX-1.40.24', 'creator': 'LaTeX with Beamer class', 'creationdate': '2023-07-28T08:08:51+05:30', 'author': 'T. V. Ramanathan', 'title': 'Modeling Univariate Time Series', 'subject': '', 'keywords': '', 'moddate': '2023-07-28T08:08:51+05:30', 'trapped': '/False', 'ptex.fullbanner': 'This is MiKTeX-pdfTeX 4.11.0 (1.40.24)', 'source': '..\\data\\Time series\\Uni_Var_Time_Series_Lecture_1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'Uni_Var_Time_Series_Lecture_1.pdf', 'file_type': 'pdf'}


In [125]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [126]:
class EmbeddingManager:
    def __init__(self,model_name: str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dim:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            raise
    
    def generate_embeddings(self,text: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ---")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model_get_sentence_embedding_dimension()



In [127]:
embeddingmanager=EmbeddingManager()
embeddingmanager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1013.98it/s]


Model loaded successfully.Embedding dim:384


C:\Users\shxpat\AppData\Local\Temp\ipykernel_30780\3508974203.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.Embedding dim:{self.model.get_sentence_embedding_dimension()}")


In [128]:
class VectorStore():
    def __init__(self,collection_name: str="pdf_documents",persist_directory: str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.clint=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata=({'discription':"PDF document embedding for RAG"})
            )
            print(f"vector store initialized Collection:{self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            

        except Exception as e:
            print(f"Error")
            raise
    
    def add_documents(self,documents: List[Any],embeddings:np.array):
        if len(documents) !=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding{len(documents)}")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)}")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print("Error")
            raise


In [129]:
vectorstore=VectorStore()
vectorstore

vector store initialized Collection:pdf_documents
Existing documents in collection: 497


In [130]:
texts=[doc.page_content for doc in chunks]

#generate the embedding 
embedding=embeddingmanager.generate_embeddings(texts)

#store in vector
vectorstore.add_documents(chunks,embedding)


Generating embeddings for 497 texts ---


Batches: 100%|██████████| 16/16 [00:08<00:00,  1.92it/s]


Generated embeddings with shape:(497, 384)
Adding497
Successfully added 497
Total documents in collection: 994


In [131]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Retrieve Query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved: {len(retrieved_docs)}")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Retrieval error: {e}")
            return []

In [132]:
rag_retriever=RAGRetriever(vectorstore,embeddingmanager)

In [133]:
rag_retriever

In [134]:
rag_retriever.retrieve("What is Stationarity?")

Retrieve Query: What is Stationarity?
Top K: 5, Score threshold: 0.0
Generating embeddings for 497 texts ---


Batches: 100%|██████████| 16/16 [00:08<00:00,  1.91it/s]


Generated embeddings with shape:(497, 384)
Retrieved: 5


[{'id': 'doc_14928738_0',
  'content': 'Modeling Univariate Time Series\nT. V. Ramanathan\nDepartment of Statistics\nSavitribai Phule Pune University\nPune - 411007 (India).\nram@unipune.ac.in\nLecture - 1\nJuly 28, 2023',
  'metadata': {'page_label': '1',
   'creator': 'LaTeX with Beamer class',
   'source': '..\\data\\Time series\\Uni_Var_Time_Series_Lecture_1.pdf',
   'total_pages': 14,
   'author': 'T. V. Ramanathan',
   'moddate': '2023-07-28T08:08:51+05:30',
   'keywords': '',
   'source_file': 'Uni_Var_Time_Series_Lecture_1.pdf',
   'doc_index': 0,
   'trapped': '/False',
   'ptex.fullbanner': 'This is MiKTeX-pdfTeX 4.11.0 (1.40.24)',
   'creationdate': '2023-07-28T08:08:51+05:30',
   'content_length': 173,
   'page': 0,
   'file_type': 'pdf',
   'producer': 'MiKTeX pdfTeX-1.40.24',
   'title': 'Modeling Univariate Time Series',
   'subject': ''},
  'similarity_score': 1.0,
  'distance': 0.0,
  'rank': 1},
 {'id': 'doc_62be4bc7_0',
  'content': 'Modeling Univariate Time Series\n

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from groq import Groq
load_dotenv()

groq_api_key =os.getenv("GROQ_API_KEY")


In [180]:
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

In [181]:
def rag_simple(query,retrieve,llm,top_k=3):

    results=retrieve.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent context"

    prompt=f"""Using the following context to answer the question concisely
    Context:
    {context}
    
    Question:{query}
    Answer:"""

    response=llm.invoke(prompt.format(context=context,query=query))

    return response.content

In [182]:
answer=rag_simple("What is Stationarity?",rag_retriever,llm)

Retrieve Query: What is Stationarity?
Top K: 3, Score threshold: 0.0
Generating embeddings for 497 texts ---


Batches: 100%|██████████| 16/16 [00:10<00:00,  1.54it/s]


Generated embeddings with shape:(497, 384)
Retrieved: 3
